# P2 Project Reimplementation — Natural Gradient Boosting

This notebook is the primary reproducible workflow for P2. The canonical implementation is the visible source tree at `code/modeling/p2_source/`. A small project-local `ngboost` facade exposes that source under the same import style used by the earlier modeling workflow. The external PyPI `ngboost` package is **not installed** by this environment and is used only through previously generated baseline tables for comparison.


## Reproducibility contract

Seed 42 is fixed. The chronological 251/77/46 match split is preserved. The evaluator derives the 128 MD1 pre-match and 189 MD1 live feature contracts from the data-pipeline audit and appends the frozen 18-feature P1 representation, yielding 146 pre-match and 207 live inputs. Hyperparameter selection, Platt calibration and uncertainty scaling use validation data only.


## 1. Locate the repository


In [ ]:
from pathlib import Path
import os, sys, subprocess
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass
candidates=[Path('/content/ML-Project-Football-Forecasting'),Path('/content/drive/MyDrive/ml_project'),Path('/content/drive/MyDrive/ML-Project-Football-Forecasting'),Path.cwd(),Path.cwd().parent]
required=Path('code/data_pipeline/results/Premier_League_2015_16_DATA_ONLY_FULL_REAL/data/gold/gold_prematch.parquet')
PROJECT_ROOT=next((p for p in candidates if (p/required).exists()),None)
if PROJECT_ROOT is None: raise FileNotFoundError('Repository not found; set PROJECT_ROOT manually.')
PROJECT_ROOT=PROJECT_ROOT.resolve(); MODELING=PROJECT_ROOT/'code/modeling'
print('PROJECT_ROOT =',PROJECT_ROOT)


## 2. Install the pinned P2 environment

The requirements intentionally omit external `ngboost`. SymPy, lifelines and tqdm are pinned because the committed source imports them.


In [ ]:
req=MODELING/'requirements_colab_p2_reimplementation.txt'
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(req)])


## 3. Verify the visible implementation

`prepare_p2_reimplementation.py` no longer reconstructs or patches an archive. It verifies the exact Python-file list and SHA-256 digest of every file in `p2_source/`, compiles the source, and checks that the local `ngboost` facade resolves successfully. Any mismatch stops the run.


In [ ]:
assert (MODELING/'p2_source').is_dir()
print('Visible P2 Python files:',len(list((MODELING/'p2_source').rglob('*.py'))))
subprocess.check_call([sys.executable,str(MODELING/'prepare_p2_reimplementation.py')])


## 4. Confirm which `ngboost` is imported

This cell makes the runtime provenance explicit before fitting anything. The module path must point inside this repository.


In [ ]:
env=os.environ.copy(); env['PYTHONPATH']=str(MODELING)
code="import ngboost,inspect; print('version=',ngboost.__version__); print('module=',inspect.getfile(ngboost)); from ngboost import NGBClassifier,NGBRegressor"
subprocess.check_call([sys.executable,'-c',code],env=env)


## 5. Mathematical and API tests

The tests check deterministic fitting, sklearn clone/Pipeline behavior, categorical probability normalization, finite-difference Normal gradients, Fisher symmetry/positive-definiteness, direct natural-gradient solves, multivariate positive-definite covariance, serialization, and distinct natural-versus-Euclidean training paths.


In [ ]:
subprocess.check_call([sys.executable,'-m','pytest','-q',str(MODELING/'test_p2_reimplementation.py')],env=env)


## 6. Task C — pre-match outcome

The P2 classifier predicts Away/Draw/Home probabilities. Candidate selection uses validation RPS. The final evidence reports raw and validation-only Platt-calibrated RPS, log loss, Brier score, ECE, accuracy, macro and per-class precision/recall/F1, confusion matrices, reliability bins, row-level predictions and ten worst cases.


## 7. Task R — pre-match margin

The P2 Normal regressor predicts final home-minus-away margin. Validation NLL selects the candidate; validation residuals determine the scale correction. Held-out evidence includes MAE, RMSE, Pearson correlation, raw/calibrated NLL, 50/80/95% coverage and interval widths, row-level means/scales/intervals and ten worst errors.


## 8. Task L — causal live forecasting

Every live snapshot uses only information available by that cutoff. Classification and margin regression are compared against a frozen P2 pre-match prediction at every minute. The runner writes aggregate, minute-level and phase-level metrics, raw/calibrated classification reliability, row-level live predictions, worst cases and live-versus-frozen curves.


## 9. Multivariate Gaussian paper experiment

The paper-aligned experiment predicts a conditional bivariate Gaussian over home and away goals. Candidate selection uses validation joint NLL. Evidence includes joint NLL, Energy Score, marginal and joint 80% coverage, covariance eigenvalues, predicted home-away correlation, validation-only covariance-dispersion correction and H/D/A probabilities derived from predictive samples.


## 10. Compute and memory methodology

Training wall time is measured directly. Peak RSS is sampled repeatedly in a background thread while each model fits rather than inferred from before/after RSS. Prediction latency is measured repeatedly and summarized with p50/p95/p99 batch times.


## 11. Execute the complete held-out evaluation

This is the expensive cell. Run it only after the source verification and tests above succeed.


In [ ]:
runner=MODELING/'run_p2_reimplementation_final.py'
subprocess.check_call([sys.executable,str(runner),'--mode','full','--project-root',str(PROJECT_ROOT)],env=env)


## 12. Unified P2 versus official-library comparison

The official PyPI NGBoost results are read only after the P2 run and are clearly labelled as a historical library baseline. The comparison covers Task C, Task R, Task L outcome and Task L margin; multivariate comparison is included when the historical table exists.


In [ ]:
import pandas as pd
from IPython.display import display,Markdown
OUT=PROJECT_ROOT/'code/modeling/outputs/p2_reimplementation_full'; TAB=OUT/'tables'
display(pd.read_csv(TAB/'p2_vs_official_library.csv'))


## 13. Review Task C/R/L evidence


In [ ]:
for name in ['p2_task_c_metrics.csv','p2_task_r_metrics.csv','p2_task_l_outcome_metrics.csv','p2_task_l_margin_metrics.csv','p2_task_l_outcome_by_phase.csv','p2_task_l_margin_by_phase.csv','p2_multivariate_metrics.csv','p2_compute_and_peak_memory.csv']:
    p=TAB/name
    if p.exists(): print('\n###',name); display(pd.read_csv(p))


## 14. Strict completion checklist and report insert

The checklist is derived from generated evidence files and requires separate C/R/L official-library comparisons; it is not a collection of unconditional `True` flags.


In [ ]:
print((OUT/'p2_completion_checklist.json').read_text())
display(Markdown((OUT/'P2_REPORT_INSERT.md').read_text(encoding='utf-8')))


## Interpretation and limitations

Do not report P2 numerical claims before this notebook completes. The held-out set contains 46 matches, so ranking differences are subject to sampling uncertainty. Snapshot rows within a match are repeated observations, and Gaussian margin/goal assumptions can remain imperfectly calibrated. The official-library comparison is descriptive evidence, not proof that the two codebases are mathematically identical.
